# 🔥 TextForge AI — LSTM Text Generation Model

**Project:** TextForge AI — Generative Text Model  
**Model:** Character-level LSTM (Long Short-Term Memory)  
**Framework:** PyTorch  

---

## What is an LSTM?

An **LSTM (Long Short-Term Memory)** is a type of Recurrent Neural Network (RNN) specifically designed to learn **long-range dependencies** in sequential data like text.

Unlike simple RNNs, LSTMs solve the **vanishing gradient problem** using three gates:
- **Forget gate** — decides what information to discard from memory
- **Input gate** — decides what new information to store
- **Output gate** — decides what to output from memory

This makes LSTMs excellent at learning patterns in text and generating coherent sequences.

---

## Architecture Overview

```
Input characters → Embedding → LSTM layers → Linear → Softmax → Next character
```

We train character-by-character, then generate text by repeatedly predicting the next character.

## Step 1: Install & Import Dependencies

In [ ]:
# Install required packages
!pip install torch numpy matplotlib tqdm -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
import os

# Use GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## Step 2: Training Data

We'll use a built-in sample corpus about Artificial Intelligence.  
In a real project, you'd load a much larger text file (e.g. Wikipedia dump, novels, etc.).

In [ ]:
# Sample training corpus — AI/Technology focused
# In production, replace this with a large text file:
# with open('your_corpus.txt', 'r') as f: text = f.read()

TEXT_CORPUS = """
Artificial intelligence is transforming the way we interact with technology and the world around us.
Machine learning algorithms enable computers to learn from data and improve their performance over time.
Deep learning, a subset of machine learning, uses neural networks with many layers to model complex patterns.
Natural language processing allows machines to understand, interpret, and generate human language.
Computer vision enables machines to interpret and understand visual information from the world.
Reinforcement learning trains agents to make decisions by rewarding desired behaviors.
Transformers have revolutionized natural language processing with their attention mechanisms.
Large language models are trained on vast amounts of text data to generate coherent and contextual responses.
The future of artificial intelligence holds promise for solving complex global challenges.
Ethical AI development requires careful consideration of bias, fairness, and transparency.
Neural networks are inspired by the biological structure of the human brain and nervous system.
Generative models can create new content including text, images, music, and video.
Transfer learning allows models trained on one task to be adapted for different but related tasks.
Autonomous systems powered by AI are being developed for transportation, healthcare, and manufacturing.
The intersection of AI and healthcare is producing breakthrough tools for diagnosis and drug discovery.
Quantum computing combined with AI could unlock computational capabilities beyond current imagination.
Data is often called the fuel of artificial intelligence and machine learning systems.
Feature engineering is the process of transforming raw data into informative representations for models.
Overfitting occurs when a model learns the training data too well and fails to generalize to new data.
Regularization techniques help prevent overfitting by penalizing model complexity.
Gradient descent is the fundamental optimization algorithm used to train neural networks.
Backpropagation computes gradients by propagating error signals through the network layers.
Activation functions introduce non-linearity into neural networks enabling them to learn complex functions.
Convolutional neural networks excel at processing grid-like data structures such as images.
Recurrent neural networks are designed to work with sequential data like text and time series.
Attention mechanisms allow models to focus on relevant parts of the input when generating output.
The softmax function converts raw model outputs into probability distributions over classes.
Embeddings represent discrete objects like words as dense vectors in a continuous space.
Hyperparameter tuning is the process of finding the optimal configuration for a machine learning model.
Cross-validation is a technique for evaluating how well a model generalizes to independent datasets.
""".strip()

print(f'Corpus length: {len(TEXT_CORPUS)} characters')
print(f'Unique characters: {len(set(TEXT_CORPUS))}')
print(f'Sample: ...{TEXT_CORPUS[100:200]}...')

## Step 3: Build Vocabulary & Encode Text

We convert characters → integers so the model can process them as tensors.

In [ ]:
# Build character vocabulary
chars     = sorted(set(TEXT_CORPUS))
vocab_size = len(chars)

# Character ↔ Integer mappings
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}
idx_to_char = {idx: ch  for idx, ch  in enumerate(chars)}

# Encode entire corpus as integers
encoded = np.array([char_to_idx[c] for c in TEXT_CORPUS])

print(f'Vocabulary size : {vocab_size} unique characters')
print(f'Encoded length  : {len(encoded)}')
print(f'First 30 chars  : {TEXT_CORPUS[:30]}')
print(f'Encoded sample  : {encoded[:30]}')

## Step 4: Create PyTorch Dataset

We create sequences of length `seq_length`. The model learns to predict the **next character** given the previous ones.

In [ ]:
class TextDataset(Dataset):
    """
    Splits the corpus into (input_sequence, target_sequence) pairs.
    For each window of seq_length chars, the target is the same
    window shifted one position to the right.
    
    Example (seq_length=5):
      input:  'artif'
      target: 'rtifi'
    """
    def __init__(self, data: np.ndarray, seq_length: int):
        self.data       = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx     : idx + self.seq_length],     dtype=torch.long)
        y = torch.tensor(self.data[idx + 1 : idx + self.seq_length + 1], dtype=torch.long)
        return x, y


SEQ_LENGTH = 80
BATCH_SIZE = 32

dataset    = TextDataset(encoded, SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f'Total sequences  : {len(dataset)}')
print(f'Batches per epoch: {len(dataloader)}')

# Peek at one batch
sample_x, sample_y = next(iter(dataloader))
print(f'Batch X shape    : {sample_x.shape}  (batch, seq_length)')
print(f'Batch Y shape    : {sample_y.shape}  (batch, seq_length)')

## Step 5: Define the LSTM Model

Our model has three parts:
1. **Embedding layer** — maps each character index to a dense vector
2. **LSTM layers** — learns sequential patterns with memory
3. **Linear (FC) layer** — projects LSTM output to vocabulary size

In [ ]:
class LSTMTextModel(nn.Module):
    """
    Character-level LSTM for text generation.
    
    Architecture:
      Embedding(vocab_size, embed_dim)
        → LSTM(embed_dim, hidden_dim, num_layers, dropout)
          → Dropout
            → Linear(hidden_dim, vocab_size)
    """
    def __init__(
        self,
        vocab_size  : int,
        embed_dim   : int = 128,
        hidden_dim  : int = 256,
        num_layers  : int = 2,
        dropout     : float = 0.3,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(
            input_size   = embed_dim,
            hidden_size  = hidden_dim,
            num_layers   = num_layers,
            batch_first  = True,
            dropout      = dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        # x shape: (batch, seq_length)
        embedded = self.embedding(x)                  # (batch, seq, embed_dim)
        out, hidden = self.lstm(embedded, hidden)      # (batch, seq, hidden_dim)
        out = self.dropout(out)
        logits = self.fc(out)                          # (batch, seq, vocab_size)
        return logits, hidden

    def init_hidden(self, batch_size: int):
        """Initialise hidden and cell states to zeros."""
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        return (h, c)


# Instantiate model
model = LSTMTextModel(
    vocab_size = vocab_size,
    embed_dim  = 128,
    hidden_dim = 256,
    num_layers = 2,
    dropout    = 0.3,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal trainable parameters: {total_params:,}')

## Step 6: Train the Model

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────
EPOCHS        = 30       # Increase to 100+ for better results
LEARNING_RATE = 0.002
CLIP          = 5.0      # Gradient clipping prevents exploding gradients

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# ── Training Loop ────────────────────────────────────────────
train_losses = []

print(f'Training on {device} for {EPOCHS} epochs...')
print('─' * 55)

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        logits, _ = model(batch_x)

        # Reshape for cross-entropy: (batch*seq, vocab) vs (batch*seq)
        loss = criterion(
            logits.view(-1, vocab_size),
            batch_y.view(-1)
        )

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping — essential for RNNs
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()

    avg_loss = epoch_loss / len(dataloader)
    train_losses.append(avg_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.5f}')

print('─' * 55)
print('Training complete!')

## Step 7: Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, color='#6366f1', linewidth=2, label='Training Loss')
plt.fill_between(range(1, EPOCHS + 1), train_losses, alpha=0.1, color='#6366f1')
plt.xlabel('Epoch',     fontsize=12)
plt.ylabel('Loss',      fontsize=12)
plt.title('TextForge AI — LSTM Training Loss', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Loss curve saved to training_loss.png')

## Step 8: Generate Text with the Trained Model

We use **temperature sampling** to control creativity:
- **Low temperature (0.3–0.5)** → more predictable, conservative output
- **High temperature (0.9–1.2)** → more random, creative output

In [ ]:
def generate_text(
    model        : LSTMTextModel,
    seed_text    : str,
    num_chars    : int   = 300,
    temperature  : float = 0.7,
) -> str:
    """
    Generate text character-by-character from a seed prompt.
    
    Args:
        model       : Trained LSTM model
        seed_text   : Starting text to prime the model
        num_chars   : Number of characters to generate
        temperature : Sampling temperature (lower=focused, higher=creative)
    """
    model.eval()
    generated = seed_text

    # Encode seed text
    input_seq = torch.tensor(
        [char_to_idx.get(c, 0) for c in seed_text],
        dtype=torch.long
    ).unsqueeze(0).to(device)  # shape: (1, seed_len)

    hidden = model.init_hidden(1)

    with torch.no_grad():
        # Prime the hidden state with seed_text
        if len(seed_text) > 1:
            _, hidden = model(input_seq[:, :-1], hidden)

        # Generate one character at a time
        current_input = input_seq[:, -1:]

        for _ in range(num_chars):
            logits, hidden = model(current_input, hidden)

            # Apply temperature scaling
            logits = logits[:, -1, :] / temperature
            probs  = torch.softmax(logits, dim=-1)

            # Sample from the distribution
            next_char_idx = torch.multinomial(probs, num_samples=1).item()
            next_char     = idx_to_char[next_char_idx]

            generated    += next_char
            current_input = torch.tensor([[next_char_idx]], dtype=torch.long).to(device)

    return generated


print('Generate function ready!')

## Step 9: Demo — Generate Text on Different Topics

In [ ]:
# ── Demo Prompts ─────────────────────────────────────────────
prompts = [
    ("Artificial intelligence", 0.5, "Conservative"),
    ("Neural networks",         0.7, "Balanced"),
    ("The future of",           0.9, "Creative"),
]

print('=' * 60)
print('  TextForge AI — LSTM Text Generation Demo')
print('=' * 60)

for seed, temp, label in prompts:
    result = generate_text(model, seed, num_chars=250, temperature=temp)
    print(f'\n📝 Prompt       : "{seed}"')
    print(f'🌡  Temperature   : {temp} ({label})')
    print(f'───────────────────────────────────────────────────')
    print(result)
    print()

## Step 10: Interactive Generator — Try Your Own Prompt!

In [ ]:
# ── Customise these values ────────────────────────────────────
YOUR_PROMPT     = "Machine learning"  # ← Change this to any topic!
YOUR_TEMP       = 0.7                 # ← 0.3 (focused) to 1.2 (creative)
YOUR_NUM_CHARS  = 400                 # ← How many characters to generate

result = generate_text(
    model,
    seed_text   = YOUR_PROMPT,
    num_chars   = YOUR_NUM_CHARS,
    temperature = YOUR_TEMP,
)

print(f'Prompt: "{YOUR_PROMPT}"')
print(f'Temperature: {YOUR_TEMP}')
print('─' * 50)
print(result)

## Step 11: Save & Load the Model

In [ ]:
# Save the trained model
checkpoint = {
    'model_state_dict' : model.state_dict(),
    'char_to_idx'      : char_to_idx,
    'idx_to_char'      : idx_to_char,
    'vocab_size'       : vocab_size,
    'hidden_dim'       : 256,
    'num_layers'       : 2,
    'embed_dim'        : 128,
}

torch.save(checkpoint, 'textforge_lstm.pth')
print('Model saved to textforge_lstm.pth')

# ── To reload later ───────────────────────────────────────────
# checkpoint  = torch.load('textforge_lstm.pth')
# model       = LSTMTextModel(checkpoint['vocab_size'], ...)
# model.load_state_dict(checkpoint['model_state_dict'])
# char_to_idx = checkpoint['char_to_idx']
# idx_to_char = checkpoint['idx_to_char']

## Summary: LSTM vs Gemini API

| Feature | LSTM (this notebook) | Gemini 2.0 Flash (TextForge web app) |
|---|---|---|
| Training data | Small custom corpus | Trillions of tokens |
| Output quality | Basic patterns | Human-level coherence |
| Customisation | Full control | Via prompt engineering |
| Cost | Free (local compute) | Free API tier |
| Speed | Fast after training | Sub-second streaming |
| Use case | Learning & research | Production apps |

---

Both approaches are valuable — the LSTM demonstrates **how text generation works under the hood**, while the Gemini API powers the **production-grade TextForge web application**.

**TextForge AI** uses Gemini for the web app because it delivers far superior output quality. This notebook exists to show understanding of the fundamental model architecture behind all modern LLMs.